In [1]:
import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
from pymoo.indicators.hv import HV
from quantum_energy.problem.modeller import RedispatchModel
from quantum_energy.problem.qubo import RedispatchQUBO
from quantum_energy.problem.redispatch_stats import RedispatchStats
from quantum_energy.moo import filter_and_sort_pf
from quantum_energy.util import get_n_switches_mat

In [2]:
def get_sim_results(Rd):
    cost_mwh = Rd.prod_cost_multi
    return {
            "t": list(range(Rd.T)),
            "powers": Rd.prod_target,
            "costs": Rd.prod_resos.T @ cost_mwh,
            "overloads": np.sum(
                Rd.S_resos @ Rd.sim_res.reso_prods
                + Rd.S_sgens @ Rd.sim_res.sgen_prods
                + Rd.S_loads @ Rd.sim_res.load_prods
                > Rd.max_load,
                axis=0,
            ),
            "switches": get_n_switches_mat(Rd.prod_resos.T),
        }

In [7]:
def get_row_per_stat(Rd, x):
    Rd_stats = RedispatchStats(Rd)
    row_per_stat = {k: np.full((1, 8), np.nan) for k in ['powers', 'costs', 'overloads', 'switches']}
    sim_res = get_sim_results(Rd)
    print(x)
    sol_res = Rd_stats.x_sol_to_stats_over_time(x)
    for t in range(len(sol_res['t'])):
        row_per_stat['powers'][0,t] = sol_res['powers'][t] * 100
        row_per_stat['overloads'][0,t] = ((sol_res['overloads'][t]/sim_res['overloads'][t]) - 1)*100
        row_per_stat['costs'][0,t] = ((sol_res['costs'][t]/sim_res['costs'][t]) - 1)*100
        if t>0:
            row_per_stat['switches'][0,t] = ((sol_res['switches'][t]/sim_res['switches'][t]) - 1)*100
    return row_per_stat

In [9]:
def get_mat_per_stat(scaler='norm'):
    rows = {"powers": [], "costs": [], "overloads": [], "switches": []}
    with open(f"../new_results/{scaler}_xs.pkl", 'rb') as f:
        xs = pickle.load(f)
    for t in [2,4,6,8]:
        T = f'T{t}'
        with open(f"../data/models/{T}.pkl", "rb") as f:
            Rd = pickle.load(f)
        row_per_stat = get_row_per_stat(Rd, xs[T])
        for k, v in row_per_stat.items():
            rows[k].append(v)
    mats = {k: np.vstack(v) for k,v in rows.items()}
    return mats

In [5]:
def plot_mats(mats, scaler='norm'):
    figure, axes = plt.subplots(2,2, figsize=(5.5,3.6))
    titles = {
        "powers": "abs(Power Diff.%)",
        "overloads": "Num. Overloads Diff.%",
        "costs": "Cost Diff.%",
        "switches": "Num. Switches Diff.%"
    }
    c = 0
    for k, v in mats.items():
        i = c % 2
        j = c // 2
        if k == 'powers':
            sns.heatmap(np.abs(v), center=0, ax=axes[i][j], cmap='icefire')
        else:
            sns.heatmap(v, center=0, ax=axes[i][j], cmap='icefire')
        axes[i][j].set_title(titles[k])
        if j == 0:
            axes[i][j].set_ylabel("Prob. Size (T)")
        if i == 1:
            axes[i][j].set_xlabel("Timepoint (t)")
        c += 1
    plt.tight_layout()
    plt.savefig(f'./plots/apply-{scaler}.pdf')
    plt.show()

In [12]:
get_mat_per_stat("norm")

[0. 0. 0. ... 0. 0. 1.]
[0. 0. 0. ... 0. 0. 1.]


KeyError: 'T6'

In [ ]:
norm_mats = {}
for i in range(10):
    cur_norm_mats = get_mat_per_stat(i, "norm")
    for k, v in cur_norm_mats.items():
        if k in norm_mats:
            norm_mats[k] = norm_mats[k] + v
        else:
            norm_mats[k] = v
for k, v in norm_mats.items():
    norm_mats[k] = v / 10
plot_mats(norm_mats, "none")

In [ ]:
none_mats = {}
for i in range(10):
    cur_none_mats = get_mat_per_stat(i, "none")
    for k, v in cur_none_mats.items():
        if k in none_mats:
            none_mats[k] = none_mats[k] + v
        else:
            none_mats[k] = v
for k, v in none_mats.items():
    none_mats[k] = v / 10
plot_mats(none_mats, "none")

In [ ]:
def plot_both(norm_mats, none_mats):
    figure, axes = plt.subplots(4,2, figsize=(6,7))
    i = 0
    for k in none_mats.keys():
        norm_hm = norm_mats[k]
        none_hm = none_mats[k]
        vmin = min(np.nanmin(norm_hm), np.nanmin(none_hm))
        vmax = max(np.nanmax(norm_hm), np.nanmax(none_hm))
        sns.heatmap(none_hm, center=0, ax=axes[i,0], vmin=vmin, vmax=vmax)
        sns.heatmap(norm_hm, center=0, ax=axes[i,1], vmin=vmin, vmax=vmax)
        axes[i,0].set_title(f'{k} - Original')
        axes[i,0].set_ylabel("Prob. Size (T)")
        if i == 3:
            axes[i,0].set_xlabel("Time point (t)")
            axes[i,1].set_xlabel("Time point (t)")
        i += 1
    plt.tight_layout()
    plt.savefig('./plots/apply.pdf')
    plt.show()

In [ ]:
plot_both(norm_mats, none_mats)